# Supply Chain Early Warning Demo

## Problem

A small supply network (3 suppliers → 4 warehouses → 5 retail nodes)
operates near steady state. At t=150, warehouse W1 — the critical
bottleneck serving retail nodes R1 and R2 — suffers a severe capacity
drop from 200 to 20 units. Downstream retail nodes begin accumulating
backlog. The cascade propagates as upstream suppliers continue producing
into a blocked network.

The challenge: detect this cascade before total system backlog crosses
the failure threshold of 200 units.

## Detection Approach

Rather than monitoring backlog directly, we track the **relational
regime indicator**:

```
I = P_max - D_max
```

Where:

- `D_max` = maximum edge strain (last dispatch / throughput limit)
- `P_max` = minimum node plasticity (spare capacity / capacity)
- `I > 0` — stable: plasticity exceeds strain
- `I = 0` — critical boundary
- `I < 0` — divergent: strain exceeds available plasticity

When I crosses zero the network is structurally divergent.
This crossing precedes observable backlog accumulation by a
measurable lead time — providing early warning without statistical
inference, sliding windows, or tuning parameters.

This is an instance of the invariant relational operator:

```
delta_M = clip(E - M, P_max)
```

The stability boundary I = 0 is where the bounded update operator
can no longer absorb incoming strain.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt

from sc_sim.network     import build_network
from sc_sim.disruption  import capacity_drop
from sc_sim.flow        import simulate
from sc_sim.instability import regime_series, compute_trigger_times

%matplotlib inline

T                  = 400
T_DISRUPTION       = 150
DISRUPTION_NODE    = 'W1'
NEW_CAP            = 20.0
BACKLOG_THRESHOLD  = 200.0

## Simulation

- 3 suppliers produce 25 units/step each (total: 75/step)
- 5 retail nodes consume 72 units/step total
- Net surplus of 3 units/step keeps backlog at zero pre-disruption
- Pipelines pre-filled to steady-state — no cold-start transient
- At t=150, W1 capacity drops from 200 to 20

In [ ]:
G   = build_network()
dfn = capacity_drop(DISRUPTION_NODE, NEW_CAP, T_DISRUPTION)
backlog, G_states = simulate(G, T=T, disruption_fn=dfn)

I_series, D_series, P_series = regime_series(G_states)
instability_t, failure_t, lead_time = compute_trigger_times(
    backlog, I_series, backlog_threshold=BACKLOG_THRESHOLD
)

## Detection Result

In [ ]:
print(f"Disruption applied:    t = {T_DISRUPTION}")
print(f"Regime I < 0 at:       t = {instability_t}  (divergent)")
print(f"Failure threshold:     t = {failure_t}")
print(f"Lead time:             {lead_time} steps before failure")

## Visualization

Four panels:

1. **Total backlog** — the conventional failure signal. Rises only after
   the cascade is already underway.

2. **Regime indicator I** — zero crossing marks structural warning.
   Fires before backlog is visible.

3. **D_max and P_max** — the relational components. D_max rises as
   edges approach capacity. P_max falls as spare inventory depletes.

4. **Regime classification** — stable / divergent zones with lead
   time annotated between warning and failure.

In [ ]:
t_axis = np.arange(T)
fig, axes = plt.subplots(4, 1, figsize=(11, 13), sharex=True)
fig.suptitle(
    'Supply Chain Early Warning — Relational Regime Indicator\n'
    r'$I = P_{\max} - D_{\max}$   '
    r'$I > 0$ stable   $I = 0$ critical   $I < 0$ divergent',
    fontsize=12
)

ax = axes[0]
ax.plot(t_axis, backlog, color='steelblue', lw=1.5, label='Total backlog')
ax.axhline(BACKLOG_THRESHOLD, color='crimson', lw=1.2, ls='--',
           label=f'Failure threshold ({BACKLOG_THRESHOLD})')
ax.axvline(T_DISRUPTION, color='orange', lw=1.0, ls=':',
           label=f'Disruption t={T_DISRUPTION}')
if failure_t is not None:
    ax.axvline(failure_t, color='crimson', lw=1.0, ls=':',
               label=f'Failure t={failure_t}')
ax.set_ylabel('Backlog (units)')
ax.legend(fontsize=8, loc='upper left')
ax.set_title('Total System Backlog', fontsize=10)

ax = axes[1]
ax.plot(t_axis, I_series, color='darkorchid', lw=1.5, label='I = P_max - D_max')
ax.axhline(0.0, color='black', lw=0.8, ls='--', label='Critical boundary (I = 0)')
ax.axvline(T_DISRUPTION, color='orange', lw=1.0, ls=':')
if instability_t is not None:
    ax.axvline(instability_t, color='darkorchid', lw=1.0, ls=':',
               label=f'I < 0 at t={instability_t}')
if failure_t is not None:
    ax.axvline(failure_t, color='crimson', lw=1.0, ls=':')
ax.fill_between(t_axis, I_series, 0, where=(I_series < 0),
                alpha=0.15, color='crimson', label='Divergent region')
ax.fill_between(t_axis, I_series, 0, where=(I_series >= 0),
                alpha=0.10, color='green',  label='Stable region')
ax.set_ylabel('I (regime indicator)')
ax.legend(fontsize=8, loc='upper left')
ax.set_title('Regime Indicator  I = P_max - D_max', fontsize=10)

ax = axes[2]
ax.plot(t_axis, D_series, color='tomato',   lw=1.3, label='D_max (strain)')
ax.plot(t_axis, P_series, color='seagreen', lw=1.3, label='P_max (spare capacity)')
ax.axvline(T_DISRUPTION, color='orange', lw=1.0, ls=':')
ax.set_ylabel('Normalised magnitude')
ax.legend(fontsize=8, loc='upper left')
ax.set_title('Relational Components', fontsize=10)

ax = axes[3]
ax.fill_between(t_axis, -1, 1, where=(I_series >= 0),
                alpha=0.25, color='green',  label='Stable  (I > 0)')
ax.fill_between(t_axis, -1, 1, where=(I_series < 0),
                alpha=0.25, color='crimson', label='Divergent (I < 0)')
ax.plot(t_axis, np.zeros(T), color='black', lw=0.8, ls='--')
ax.axvline(T_DISRUPTION, color='orange', lw=1.0, ls=':',
           label=f'Disruption t={T_DISRUPTION}')
if instability_t is not None:
    ax.axvline(instability_t, color='darkorchid', lw=1.0, ls=':',
               label=f'Warning t={instability_t}')
if failure_t is not None:
    ax.axvline(failure_t, color='crimson', lw=1.0, ls=':',
               label=f'Failure t={failure_t}')
if lead_time is not None:
    mid = instability_t + lead_time // 2
    ax.annotate(
        f'{lead_time} step lead',
        xy=(failure_t, 0), xytext=(mid, 0.6),
        arrowprops=dict(arrowstyle='->', color='black', lw=0.8),
        fontsize=8, ha='center'
    )
ax.set_ylim(-1.2, 1.2)
ax.set_ylabel('Regime zone')
ax.set_xlabel('Time step')
ax.legend(fontsize=8, loc='upper left')
ax.set_title('Regime Classification', fontsize=10)

plt.tight_layout()
plt.show()

## Sensitivity Analysis

Lead time versus disruption severity — sweeping W1 new capacity
from 5 (near-total failure) to 60 (mild reduction).
Demonstrates that the regime indicator detects structural divergence
across a wide range of disruption magnitudes without retuning.

In [ ]:
cap_values = np.arange(5, 65, 5)
lead_times = []

for cap in cap_values:
    g   = build_network()
    d   = capacity_drop(DISRUPTION_NODE, float(cap), T_DISRUPTION)
    bl, gs = simulate(g, T=T, disruption_fn=d)
    I_, _, _ = regime_series(gs)
    _, _, lt = compute_trigger_times(bl, I_,
                                     backlog_threshold=BACKLOG_THRESHOLD)
    lead_times.append(lt if lt is not None else 0)

plt.figure(figsize=(9, 4))
plt.plot(cap_values, lead_times, 'o-', color='steelblue', lw=1.5)
plt.xlabel('W1 New Capacity (post-disruption)')
plt.ylabel('Lead Time (steps)')
plt.title('Regime Detection — Lead Time vs Disruption Severity')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()